# BO-IGBT microseismic classification

This version is based on `BO_IGBT_revised.py` and adds **train/validation/test** metrics, per-sample outputs, classification reports, confusion matrices, ROC/PR curves, base-model feature importances, and optional **stacking SHAP**.

**Required input**: `Data_sets/MS_NS_TrainData/Features_matrix.mat` (`Features_matrix`: N×P) and `Label.mat` (`Label`: N×1, `1=MS, 0=NS`). Recommended optional file: `Group.mat` (`Group`: N event IDs). Optional `Feature_names.txt` contains one feature name per line. Independent site (if available): `Data_sets/Independent_Tunnel/{Features_matrix.mat,Label.mat}`.

Install: `pip install numpy scipy pandas matplotlib scikit-learn xgboost lightgbm catboost bayesian-optimization shap nbformat`.

Outputs are written to `Results_BO_IGBT/`. **Set `RUN_SHAP=True`** in configuration to obtain model-agnostic SHAP, which can be computationally expensive.

In [2]:
from pathlib import Path
import os
import json
import warnings
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.special import expit, logit
from sklearn.model_selection import (train_test_split, GroupShuffleSplit,
                                     StratifiedKFold, StratifiedGroupKFold)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, balanced_accuracy_score,
                             confusion_matrix, classification_report, matthews_corrcoef,
                             roc_curve, precision_recall_curve)
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

SEED = 42
ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
DATA_DIR = ROOT / 'Data_sets' / 'MS_NS_TrainData'
OUTPUT_DIR = ROOT / os.environ.get('BO_IGBT_OUTPUT', 'Results_BO_IGBT')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Dataset: small programming example.
EXAMPLE_21 = True
ALLOW_SYNTHETIC_IF_MISSING = True
EXPECTED_MS, EXPECTED_NS = 7, 14

# BO (TPE/Optuna) tunes ONLY the stacking branch.
ENABLE_BAYES_OPT = True
ENSEMBLE_NAME = 'BO-IGBT'
N_INIT = int(os.environ.get('BO_INIT_TRIALS', 3))
N_BO_ITER = int(os.environ.get('BO_ADDITIONAL_TRIALS', 9))  # default 12 trials / learner
N_FOLDS = 3
ALPHA_POS = 0.75  # 1 = MS, 0 = NS
GAMMA = 2.0
THRESHOLD = 0.5
META_THRESHOLD_MODE = 'validation_f1'  # selects meta C/class_weight/threshold using validation ONLY
PASSTHROUGH = False

# Importance via requested built-in interfaces always runs.
RUN_EXPLANATIONS = False       # Optional permutation importance on tiny TEST is unstable.
RUN_SHAP = False
PERMUTATION_REPEATS = 8
MAX_PERMUTATION_SAMPLES = 1200
SHAP_BACKGROUND_SIZE = 10
SHAP_EXPLAIN_SIZE = 4
TOP_K_FEATURES = 20
IMAGE_DPI = int(os.environ.get('BO_IMAGE_DPI', 600))




In [3]:
def read_mat(path, preferred_var=None):
    if not Path(path).is_file():
        raise FileNotFoundError(f'Missing required file: {path}')
    obj = sio.loadmat(str(path))
    if preferred_var in obj:
        return np.asarray(obj[preferred_var])
    vals = [v for k, v in obj.items() if not k.startswith('__') and isinstance(v, np.ndarray)]
    if len(vals) != 1:
        raise ValueError(f'{path}: ambiguous MAT variables; specify {preferred_var}')
    return vals[0]


def load_dataset(folder):
    folder = Path(folder)
    fx, fy = folder / 'Features_matrix.mat', folder / 'Label.mat'
    if fx.is_file() and fy.is_file():
        X = np.asarray(read_mat(fx, 'Features_matrix'), dtype=float)
        y = np.asarray(read_mat(fy, 'Label')).reshape(-1).astype(int)
        print(f'Loaded REAL data: {fx} and {fy}')
    elif ALLOW_SYNTHETIC_IF_MISSING and not fx.exists() and not fy.exists():
        rng = np.random.default_rng(SEED)
        y = np.r_[np.ones(EXPECTED_MS, int), np.zeros(EXPECTED_NS, int)]
        X = rng.normal(size=(len(y), 12))
        # Illustrative class differences, NOT measured microseismic features.
        X[y == 1, :3] += np.array([1.4, 0.8, -1.0])
        perm = rng.permutation(len(y))
        X, y = X[perm], y[perm]
        warnings.warn('SYNTHETIC DEMO DATA GENERATED. '
                      'All results are illustrative, NOT real MS classification results.')
        folder = None
    else:
        raise FileNotFoundError(f'Expected both files: {fx} and {fy}')

    if X.ndim != 2 or X.shape[0] != len(y):
        raise ValueError('Expected feature matrix N×P and matching N labels')
    if set(np.unique(y).tolist()) != {0, 1}:
        raise ValueError('Labels must be binary 1=MS and 0=NS')
    if EXAMPLE_21 and (len(y) != 21 or (y == 1).sum() != EXPECTED_MS or
                       (y == 0).sum() != EXPECTED_NS):
        raise ValueError(f'Expected exactly 21 samples: MS={EXPECTED_MS}, '
                         f'NS={EXPECTED_NS}; got N={len(y)}, '
                         f'MS={sum(y==1)}, NS={sum(y==0)}')
    groups = None
    if folder is not None:
        gfile = folder / 'Group.mat'
        if gfile.is_file():
            groups = np.asarray(read_mat(gfile, 'Group')).reshape(-1)
            if len(groups) != len(y):
                raise ValueError('Group.mat requires one original-event ID per row')
    print(f'Samples={len(y)} | MS={sum(y==1)} | NS={sum(y==0)} | features={X.shape[1]}')
    return X, y, groups


# The real uses 80/10/10 train/validation/test

def split_data(X, y, groups=None, random_state=SEED):
    ix = np.arange(len(y))
    if EXAMPLE_21 and groups is None:
        # EXACT per-class allocation; reproducible. Train 13, valid 4, test 4.
        rng = np.random.default_rng(random_state)
        pos = rng.permutation(ix[y == 1])
        neg = rng.permutation(ix[y == 0])
        tr = np.r_[pos[:5], neg[:8]]
        va = np.r_[pos[5:6], neg[8:11]]
        te = np.r_[pos[6:7], neg[11:14]]
        ix_tr, ix_va, ix_te = (rng.permutation(arr) for arr in (tr, va, te))
        warnings.warn('Sample-wise split: correlated segments of the same event '
                      'could leak across sets. Provide Group.mat where available.')
    elif groups is None:
        ix_tr, ix_tmp = train_test_split(ix, test_size=0.2, stratify=y,
                                         random_state=random_state)
        ix_va, ix_te = train_test_split(ix_tmp, test_size=0.5,
                                        stratify=y[ix_tmp], random_state=random_state)
    else:
        
        candidate = None
        for seed in range(random_state, random_state + 300):
            gss = GroupShuffleSplit(n_splits=1, test_size=0.35, random_state=seed)
            a, b = next(gss.split(X, y, groups=groups))
            if len(np.unique(y[a])) < 2 or min(np.bincount(y[a], minlength=2)) < N_FOLDS:
                continue
            gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
            c, d = next(gss2.split(X[b], y[b], groups=groups[b]))
            if len(np.unique(y[b[c]])) < 2 or len(np.unique(y[b[d]])) < 2:
                continue
            candidate = (ix[a], ix[b[c]], ix[b[d]])
            break
        if candidate is None:
            raise ValueError('Samples cannot form train/valid/test with both '
                             'classes and enough train positives for 3-fold OOF; '
                             'add independent events or adjust split design.')
        ix_tr, ix_va, ix_te = candidate
        for i1,i2 in [(ix_tr,ix_va),(ix_tr,ix_te),(ix_va,ix_te)]:
            assert not set(groups[i1]) & set(groups[i2])
    for name, ids in [('train', ix_tr), ('validation', ix_va), ('test', ix_te)]:
        if len(np.unique(y[ids])) < 2:
            raise ValueError(f'{name} lacks at least one class')
        print(f'{name:>10}: N={len(ids):2d} | MS={sum(y[ids]==1):2d} | NS={sum(y[ids]==0):2d}')
    return ix_tr, ix_va, ix_te


def new_preprocessor():
    return Pipeline([('imputer', SimpleImputer(strategy='median')),
                     ('scaler', StandardScaler())])




In [4]:
# L = -alpha_t * (1-p_t)**gamma * log(p_t)
# alpha_t = alpha_pos for y=1, (1-alpha_pos) for y=0.
class BinaryFocal:
    def __init__(self, alpha_pos=0.75, gamma=2.0, hess_floor=1e-6):
        self.alpha_pos = float(alpha_pos)
        self.gamma = float(gamma)
        self.hess_floor = float(hess_floor)

    def grad_hess(self, raw, y, weights=None, clip_hess=True):
        raw = np.asarray(raw, dtype=float).reshape(-1)
        y = np.asarray(y, dtype=int).reshape(-1)
        t = 2 * y - 1
        q = np.clip(expit(t * raw), 1e-12, 1.0 - 1e-12)
        u = 1.0 - q
        a = np.where(y == 1, self.alpha_pos, 1 - self.alpha_pos)
        g = self.gamma
        grad = a * t * u**g * (g * q * np.log(q) - u)
        hess = a * q * u**g * (u * (2*g + 1 + g*np.log(q)) - g*g*q*np.log(q))
        if weights is not None:
            w = np.asarray(weights, dtype=float).reshape(-1)
            grad *= w
            hess *= w
        if clip_hess:
            # Focal loss can be non-convex; use a nonnegative Hessian approximation.
            hess = np.maximum(hess, self.hess_floor)
        return grad, hess

    def loss(self, raw, y):
        y = np.asarray(y, dtype=int).reshape(-1)
        t = 2*y-1
        q = np.clip(expit(t * np.asarray(raw)), 1e-12, 1-1e-12)
        a = np.where(y == 1, self.alpha_pos, 1-self.alpha_pos)
        return -a * (1-q)**self.gamma * np.log(q)


FOCAL = BinaryFocal(alpha_pos=ALPHA_POS, gamma=GAMMA)


def focal_xgb(preds, dataset):
    return FOCAL.grad_hess(preds, dataset.get_label())


def focal_lgb(preds, dataset):
    return FOCAL.grad_hess(preds, dataset.get_label())


class FocalCatObjective:
    # CatBoost requires NEGATIVE first and second derivatives.
    def calc_ders_range(self, approxes, targets, weights):
        grad, hess = FOCAL.grad_hess(approxes, targets, weights=weights)
        return list(zip((-grad).tolist(), (-hess).tolist()))


def check_focal_derivatives():
    for y in [0, 1]:
        for z in [-5.0, -2.0, 0.0, 2.0, 5.0]:
            dz = 1e-4
            g, h = FOCAL.grad_hess([z], [y], clip_hess=False)
            lp = FOCAL.loss([z+dz], [y])[0]
            lm = FOCAL.loss([z-dz], [y])[0]
            l0 = FOCAL.loss([z], [y])[0]
            g_fd = (lp-lm)/(2*dz)
            h_fd = (lp-2*l0+lm)/dz**2
            assert np.isclose(g[0], g_fd, atol=2e-6)
            assert np.isclose(h[0], h_fd, atol=5e-5)
    print('Focal derivative finite-difference tests passed.')




In [5]:
def normalize_params(name, p):
    p = dict(p)
    if name == 'xgb':
        return {'max_depth': int(round(p['max_depth'])),
                'learning_rate': float(p['learning_rate']),
                'subsample': float(p['subsample']),
                'reg_lambda': float(p['reg_lambda']),
                'num_boost_round': int(round(p['num_boost_round']))}
    if name == 'lgb':
        return {'max_depth': int(round(p['max_depth'])),
                'learning_rate': float(p['learning_rate']),
                'num_leaves': int(round(p['num_leaves'])),
                'feature_fraction': float(p['feature_fraction']),
                'num_boost_round': int(round(p['num_boost_round']))}
    if name == 'cat':
        return {'depth': int(round(p['depth'])),
                'learning_rate': float(p['learning_rate']),
                'l2_leaf_reg': float(p['l2_leaf_reg']),
                'num_boost_round': int(round(p['num_boost_round']))}
    raise ValueError(name)


def fit_base(name, p, X_fit, y_fit, X_val=None, y_val=None, early_stop=False):
    rounds = int(p['num_boost_round'])
    if name == 'xgb':
        dtr = xgb.DMatrix(X_fit, label=y_fit)
        evals = []
        if X_val is not None:
            evals = [(xgb.DMatrix(X_val, label=y_val), 'valid')]
        params = {'max_depth': p['max_depth'], 'eta': p['learning_rate'],
                  'subsample': p['subsample'], 'lambda': p['reg_lambda'],
                  'base_score': 0.0, 'min_child_weight': p.get('min_child_weight', 0.02),
                  'eval_metric': 'auc', 'seed': SEED,
                  'tree_method': 'hist', 'nthread': 4,
                  'disable_default_eval_metric': 1}
        model = xgb.train(params, dtr, num_boost_round=rounds, obj=focal_xgb,
                          evals=evals,
                          early_stopping_rounds=30 if early_stop else None,
                          verbose_eval=False)
        best = model.best_iteration + 1 if early_stop else rounds
    elif name == 'lgb':
        dtr = lgb.Dataset(X_fit, label=y_fit)
        dval = lgb.Dataset(X_val, label=y_val, reference=dtr) if X_val is not None else None
        params = {'objective': focal_lgb, 'metric': 'auc',
                  'max_depth': p['max_depth'], 'learning_rate': p['learning_rate'],
                  'num_leaves': min(p['num_leaves'], 2**p['max_depth']),
                  'feature_fraction': p['feature_fraction'],
                  'min_data_in_leaf': 2 if EXAMPLE_21 else 20,
                  'min_data_in_bin': 1 if EXAMPLE_21 else 3,
                  'feature_pre_filter': False,
                  'verbosity': -1, 'seed': SEED, 'num_threads': 4}
        callbacks = [lgb.early_stopping(30, verbose=False)] if early_stop else []
        model = lgb.train(params, dtr, num_boost_round=rounds,
                          valid_sets=[dval] if dval is not None else None,
                          callbacks=callbacks)
        best = model.best_iteration if early_stop and model.best_iteration else rounds
    elif name == 'cat':
        model = CatBoostClassifier(loss_function=FocalCatObjective(),
                                   eval_metric='AUC', depth=p['depth'],
                                   learning_rate=p['learning_rate'],
                                   l2_leaf_reg=p['l2_leaf_reg'],
                                   iterations=rounds, random_seed=SEED,
                                   thread_count=4, task_type='CPU',
                                   verbose=False, allow_writing_files=False)
        fit_args = {'verbose': False}
        if X_val is not None:
            fit_args['eval_set'] = (X_val, y_val)
            if early_stop:
                fit_args['early_stopping_rounds'] = 30
                fit_args['use_best_model'] = True
        model.fit(X_fit, np.asarray(y_fit).reshape(-1), **fit_args)
        best_it = model.get_best_iteration() if early_stop else None
        best = max(1, best_it+1) if best_it is not None and best_it >= 0 else rounds
    else:
        raise ValueError(name)
    return model, best


def predict_base(name, model, X, n_trees=None):
    if name == 'xgb':
        kw = {'iteration_range': (0, int(n_trees))} if n_trees is not None else {}
        raw = model.predict(xgb.DMatrix(X), output_margin=True, **kw)
    elif name == 'lgb':
        kw = {'num_iteration': int(n_trees)} if n_trees is not None else {}
        raw = model.predict(X, raw_score=True, **kw)
    elif name == 'cat':
        kw = {'ntree_end': int(n_trees)} if n_trees is not None else {}
        raw = model.predict(X, prediction_type='RawFormulaVal', **kw)
    else:
        raise ValueError(name)
    return expit(np.asarray(raw).reshape(-1))




In [6]:
PB = {
    'xgb': {'max_depth': (2, 4), 'learning_rate': (0.01, 0.30),
            'subsample': (0.6, 1.0), 'reg_lambda': (0.1, 10.0),
            'num_boost_round': (20, 60)},
    'lgb': {'max_depth': (2, 4), 'num_leaves': (3, 12),
            'learning_rate': (0.01, 0.30), 'feature_fraction': (0.5, 1.0),
            'num_boost_round': (20, 60)},
    'cat': {'depth': (2, 4), 'learning_rate': (0.01, 0.30),
            'l2_leaf_reg': (1, 10), 'num_boost_round': (20, 60)}
}




PARAMS = {
    'xgb': {'max_depth': 2, 'learning_rate': 0.08, 'subsample': 1.0,
            'reg_lambda': 1.0, 'min_child_weight': 0.02, 'num_boost_round': 60},
    'lgb': {'max_depth': 3, 'num_leaves': 7, 'learning_rate': 0.08,
            'feature_fraction': 1.0, 'num_boost_round': 60},
    'cat': {'depth': 3, 'learning_rate': 0.08, 'l2_leaf_reg': 3.0,
            'num_boost_round': 60}
}




In [7]:
NAMES = ['xgb', 'lgb', 'cat']




def stacking_probability(ensemble, X):
    Xt = ensemble['preprocessor'].transform(X)
    proba = np.column_stack([predict_base(n, ensemble['base'][n], Xt) for n in NAMES])
    inputs = np.c_[proba, Xt] if ensemble['passthrough'] else proba
    return ensemble['meta'].predict_proba(inputs)[:, 1]


def score_report(y, prob, label, threshold=THRESHOLD):
    pred = (np.asarray(prob) >= threshold).astype(int)
    m = {'Dataset': label, 'N': len(y), 'MS_count': int(np.sum(np.asarray(y)==1)),
         'Accuracy': accuracy_score(y, pred),
         'Precision': precision_score(y, pred, zero_division=0),
         'Recall': recall_score(y, pred, zero_division=0),
         'F1': f1_score(y, pred, zero_division=0),
         'AUC': roc_auc_score(y, prob) if len(np.unique(y)) == 2 else np.nan}
    print(label, m)
    return m




In [8]:
# IMPORTANT: SNR in dB is defined on RAW SEISMIC WAVEFORMS, not tabular features.
# Create noise-perturbed waveforms FIRST and run the identical feature extractor.
def evaluate_independent_condition(ens, condition_folder, condition_name):
    Xc, yc, _ = load_dataset(condition_folder)
    return score_report(yc, stacking_probability(ens, Xc), condition_name,
                        threshold=ens['decision_threshold'])


def add_awgn_to_waveforms(waveforms, snr_db, seed=SEED):
    """Inject AWGN per waveform along last axis; re-extract features afterwards."""
    rng = np.random.default_rng(seed)
    W = np.asarray(waveforms, dtype=float)
    noise = rng.standard_normal(W.shape)
    signal_power = np.mean(W**2, axis=-1, keepdims=True)
    noise_power = np.mean(noise**2, axis=-1, keepdims=True)
    noise = noise * np.sqrt((signal_power / (10**(snr_db/10))) /
                             np.maximum(noise_power, 1e-12))
    return W + noise




In [9]:
# Hyperparameters are retuned on the restricted training pool ONLY.
def stratified_n_samples(indices, y, n, seed=SEED):
    if n > len(indices):
        raise ValueError(f'Requested {n} samples; only {len(indices)} available')
    ix = np.asarray(indices)
    picked, _ = train_test_split(ix, train_size=n, stratify=y[ix], random_state=seed)
    return picked


def ratio_1_to_32_subset(indices, y, seed=SEED):
    rng = np.random.default_rng(seed)
    ix = np.asarray(indices)
    pos = ix[y[ix] == 1]
    neg = ix[y[ix] == 0]
    n_pos = min(len(pos), len(neg)//32)
    if n_pos < 12:
        raise ValueError('Insufficient MS events for 3-fold imbalance experiment')
    selected = np.r_[rng.choice(pos, n_pos, replace=False),
                     rng.choice(neg, n_pos*32, replace=False)]
    rng.shuffle(selected)
    return selected


def run_restricted_experiment(X, y, pool_indices, test_indices, label,
                              groups=None):
    pool_indices = np.asarray(pool_indices)
    test_indices = np.asarray(test_indices)
    if np.intersect1d(pool_indices, test_indices).size:
        raise ValueError('Test samples must not be in training pool')

    restricted = np.arange(len(pool_indices))
    if groups is None:
        local_fit, local_val = train_test_split(restricted, test_size=0.20,
                 stratify=y[pool_indices], random_state=SEED)
    else:
        gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
        local_fit, local_val = next(gss.split(X[pool_indices], y[pool_indices],
                                               groups[pool_indices]))
    fit_ix, val_ix = pool_indices[local_fit], pool_indices[local_val]
    ensemble = fit_stacking(X[fit_ix], y[fit_ix], X[val_ix], y[val_ix],
                            groups_train=groups[fit_ix] if groups is not None else None)
    return score_report(y[test_indices], stacking_probability(ensemble, X[test_indices]), label,
                        threshold=ensemble['decision_threshold'])




In [10]:
# Notes:
# - Train metrics are IN-SAMPLE (refitted base learners and meta learner); optimistic.
# - Validation was used for Bayesian hyperparameter selection; not an unbiased test.

def load_feature_names(folder, n_features):
    folder = Path(folder)
    names = None
    for fname in ('Feature_names.txt', 'Feature_names.csv'):
        file = folder / fname
        if file.exists():
            # One feature name per line; CSV also supports a single Name column.
            if file.suffix == '.txt':
                names = [line.strip() for line in file.read_text(encoding='utf-8-sig').splitlines()
                         if line.strip()]
            else:
                df = pd.read_csv(file)
                names = df.iloc[:, 0].astype(str).str.strip().tolist()
            break
    if names is None:
        names = [f'F{i+1:02d}' for i in range(n_features)]
    if len(names) != n_features or len(set(names)) != len(names):
        raise ValueError('Feature_names must contain exactly one UNIQUE name per feature column')
    pd.DataFrame({'Feature_index': range(1, n_features+1),
                  'Feature': names}).to_csv(OUTPUT_DIR / 'feature_name_mapping.csv', index=False)
    return names




def metrics_one(y_true, prob, split_name, model_name, threshold=THRESHOLD):
    y_true = np.asarray(y_true, dtype=int).reshape(-1)
    prob = np.asarray(prob, dtype=float).reshape(-1)
    pred = (prob >= threshold).astype(int)
    if len(y_true) != len(prob):
        raise ValueError('Prediction length mismatch')
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    out = dict(Split=split_name, Model=model_name, N=len(y_true),
               NS=int((y_true == 0).sum()), MS=int((y_true == 1).sum()),
               Accuracy=accuracy_score(y_true, pred),
               Balanced_Accuracy=balanced_accuracy_score(y_true, pred),
               Precision_MS=precision_score(y_true, pred, zero_division=0),
               Recall_MS=recall_score(y_true, pred, zero_division=0),
               F1_MS=f1_score(y_true, pred, zero_division=0),
               Specificity_NS=tn / (tn+fp) if (tn+fp) else np.nan,
               F1_macro=f1_score(y_true, pred, average='macro', zero_division=0),
               MCC=matthews_corrcoef(y_true, pred),
               ROC_AUC=roc_auc_score(y_true, prob) if len(np.unique(y_true)) == 2 else np.nan,
               Average_Precision=average_precision_score(y_true, prob) if
                                      np.any(y_true == 1) else np.nan,
               TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp),
               Threshold=threshold)
    return out


def save_plot(fig, path_without_ext):
    path_without_ext = Path(path_without_ext)
    path_without_ext.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(path_without_ext)+'.png', dpi=IMAGE_DPI,
                bbox_inches='tight', facecolor='white')
    fig.savefig(str(path_without_ext)+'.pdf', bbox_inches='tight', facecolor='white')
    fig.savefig(str(path_without_ext)+'.eps', format='eps', bbox_inches='tight', facecolor='white')
    plt.close(fig)


def plot_confusion_panels(y_true, prob_by_model, split_name, fig_dir, thresholds=None):
    models = list(prob_by_model)
    fig, axes = plt.subplots(2, 2, figsize=(10.2, 8.2), constrained_layout=True)
    for ax, model in zip(axes.flat, models):
        th = (thresholds or {}).get(model, THRESHOLD)
        pred = (np.asarray(prob_by_model[model]) >= th).astype(int)
        cm = confusion_matrix(y_true, pred, labels=[0, 1]); row_sum = cm.sum(axis=1)
        img = ax.imshow(cm, cmap='Blues', vmin=0, vmax=max(1, cm.max()))
        for i in range(2):
            for j in range(2):
                pct = 100.0 * cm[i,j] / row_sum[i] if row_sum[i] else 0.0
                ax.text(j, i, f'{cm[i,j]}\n({pct:.1f}%)', ha='center', va='center',
                        color='white' if cm[i,j] > 0.55*cm.max() else 'black', fontsize=10)
        ax.set_xticks([0, 1], ['NS', 'MS']); ax.set_yticks([0, 1], ['NS', 'MS'])
        ax.set_xlabel('Predicted class'); ax.set_ylabel('True class')
        ax.set_title(model)
    fig.suptitle(f'{split_name}: confusion matrices', fontsize=13)
    save_plot(fig, fig_dir / f'confusion_{split_name}')


def plot_roc_pr(y_true, prob_by_model, split_name, fig_dir):
    if len(np.unique(y_true)) != 2:
        return
    fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(11.5, 4.6),
                                       constrained_layout=True)
    for model, proba in prob_by_model.items():
        proba = np.asarray(proba)
        fpr, tpr, _ = roc_curve(y_true, proba)
        precision, recall, _ = precision_recall_curve(y_true, proba)
        auroc = roc_auc_score(y_true, proba)
        ap = average_precision_score(y_true, proba)
        ax_roc.plot(fpr, tpr, lw=1.5, label=f'{model} (AUC={auroc:.3f})')
        ax_pr.plot(recall, precision, lw=1.5, label=f'{model} (AP={ap:.3f})')
        curve_path = fig_dir.parent / 'curve_data' / split_name
        curve_path.mkdir(parents=True, exist_ok=True)
        pd.DataFrame({'FPR': fpr, 'TPR': tpr}).to_csv(
            curve_path / f'ROC_{model}.csv', index=False)
        pd.DataFrame({'Recall': recall, 'Precision': precision}).to_csv(
            curve_path / f'PR_{model}.csv', index=False)
    ax_roc.plot([0, 1], [0, 1], '--', color='gray', lw=1)
    ax_roc.set(xlabel='False positive rate', ylabel='True positive rate',
               title='ROC curve', xlim=(0, 1), ylim=(0, 1.02))
    ax_pr.axhline(np.mean(y_true==1), ls='--', lw=1, color='gray', label='MS prevalence')
    ax_pr.set(xlabel='Recall (MS)', ylabel='Precision (MS)',
              title='Precision–recall curve', xlim=(0, 1), ylim=(0, 1.02))
    for ax in (ax_roc, ax_pr):
        ax.legend(fontsize=8, loc='best'); ax.grid(alpha=0.2)
    fig.suptitle(split_name)
    save_plot(fig, fig_dir / f'ROC_PR_{split_name}')


def evaluate_all_splits(ensemble, X, y, split_map, groups=None, append=False):
    # Each split gets per-sample predictions, confusion matrix, ROC and PR
    # curves, and classification reports for the three learners and ensemble.
    metrics = []
    fig_dir = OUTPUT_DIR / 'figures'
    pred_dir = OUTPUT_DIR / 'predictions'
    rep_dir = OUTPUT_DIR / 'classification_reports'
    for folder in (fig_dir, pred_dir, rep_dir): folder.mkdir(parents=True, exist_ok=True)
    for split_name, ids in split_map.items():
        ids = np.asarray(ids)
        truth = np.asarray(y[ids], dtype=int)
        probs = predict_all_models(ensemble, X[ids])
        thresholds = {n: THRESHOLD for n in NAMES}
        thresholds[ENSEMBLE_NAME] = ensemble['decision_threshold']
        plot_confusion_panels(truth, probs, split_name, fig_dir, thresholds)
        plot_roc_pr(truth, probs, split_name, fig_dir)
        for model, prob in probs.items():
            th = thresholds[model]
            result = metrics_one(truth, prob, split_name, model, threshold=th)
            metrics.append(result)
            pred = (prob >= th).astype(int)
            table = pd.DataFrame({'Row_ID_1based': ids+1, 'True_label': truth,
                                  'Predicted_label': pred, 'Probability_MS': prob,
                                  'Correct': (truth == pred).astype(int)})
            if groups is not None:
                table.insert(1, 'Group_ID', [str(g) for g in groups[ids]])
            table.to_csv(pred_dir / f'{split_name}_{model}_predictions.csv', index=False)
            report_dict = classification_report(truth, pred, labels=[0,1],
                              target_names=['NS','MS'], output_dict=True,
                              zero_division=0)
            pd.DataFrame(report_dict).T.to_csv(
                rep_dir / f'{split_name}_{model}_classification_report.csv')
            (rep_dir / f'{split_name}_{model}_classification_report.txt').write_text(
                classification_report(truth, pred, labels=[0,1],
                             target_names=['NS','MS'], zero_division=0),
                encoding='utf-8')
            print(f'{split_name:17} | {model:8} | F1_MS={result["F1_MS"]:.4f} '
                  f'| ROC_AUC={result["ROC_AUC"]:.4f} | N={len(truth)}')
    return pd.DataFrame(metrics)


def save_split_summary(y, split_map, groups=None):
    rows=[]
    for label, inds in split_map.items():
        ids = np.asarray(inds)
        rows.append({'Split':label,'N':len(ids),
                     'NS':int((y[ids]==0).sum()), 'MS':int((y[ids]==1).sum()),
                     'MS_ratio':float((y[ids]==1).mean()),
                     'N_groups':int(len(np.unique(groups[ids]))) if groups is not None else np.nan})
    pd.DataFrame(rows).to_csv(OUTPUT_DIR/'split_summary.csv',index=False)

def save_train_oof_base_results(ensemble, y_train, train_id, groups=None):
    
    oof = np.asarray(ensemble['train_oof_base'])
    y_train = np.asarray(y_train).reshape(-1)
    if len(oof) != len(y_train):
        raise ValueError('OOF base predictions do not match training samples')
    save_dir = OUTPUT_DIR / 'predictions'
    save_dir.mkdir(parents=True, exist_ok=True)
    table = pd.DataFrame({'Row_ID_1based': np.asarray(train_id)+1,
                          'True_label': y_train})
    if groups is not None:
        table['Group_ID'] = [str(g) for g in groups[train_id]]
    metrics = []
    for j, model in enumerate(NAMES):
        table[f'BO_component_{model}_OOF_score_MS'] = oof[:, j]
        table[f'BO_component_{model}_OOF_prediction'] = (oof[:, j] >= THRESHOLD).astype(int)
        metrics.append(metrics_one(y_train, oof[:, j], 'train_OOF_BO_components', f'BO_component_{model}'))
    table.to_csv(save_dir / 'train_OOF_base_probabilities.csv', index=False)
    pd.DataFrame(metrics).to_csv(OUTPUT_DIR / 'train_OOF_BO_components_metrics.csv', index=False)




In [11]:
def base_feature_importances(ensemble, feature_names):
    n_features = len(feature_names)
    rows=[]
    for model_name in NAMES:
        model=ensemble['base'][model_name]
        if model_name == 'xgb':
            # Use average gain; unselected features have zero importance.
            gain=model.get_score(importance_type='gain')
            vals=np.array([gain.get(f'f{i}', 0.0) for i in range(n_features)], float)
            method='gain'
        elif model_name == 'lgb':
            vals=np.asarray(model.feature_importance(importance_type='gain'),dtype=float)
            method='total_gain'
        else:
            vals=np.asarray(model.get_feature_importance(type='FeatureImportance'),dtype=float)
            method='PredictionValuesChange'
        if len(vals)!=n_features: raise ValueError(f'{model_name}: mismatched feature count')
        denom=max(float(np.abs(vals).sum()),1e-15)
        for j, name in enumerate(feature_names):
            rows.append({'Model':model_name,'Method':method, 'Feature':name,
                         'Feature_index':j+1, 'Raw_importance':float(vals[j]),
                         'Within_model_importance_percent':100*float(vals[j])/denom})
    return pd.DataFrame(rows)


def plot_base_importances(df, figure_dir, top_k=20):
    fig, axes=plt.subplots(1,3, figsize=(15.5,7), constrained_layout=True)
    for model_name, ax in zip(NAMES, axes):
        part=df[df['Model']==model_name].nlargest(top_k,'Within_model_importance_percent')
        part=part.iloc[::-1]
        ax.barh(part['Feature'], part['Within_model_importance_percent'])
        ax.set_title(model_name.upper()); ax.set_xlabel('Within-model importance (%)')
        ax.tick_params(axis='y', labelsize=8)
        ax.grid(axis='x', alpha=0.2)
    fig.suptitle('Base learner feature importance (methods differ; do not compare magnitudes across models)')
    save_plot(fig, figure_dir/'base_feature_importance')


def subsample_for_explanation(X, y, max_samples, seed=SEED):
    y=np.asarray(y)
    if len(X)<=max_samples: return np.asarray(X), y, np.arange(len(y))
    if np.min(np.bincount(y.astype(int),minlength=2))>=2:
        take, _ = train_test_split(np.arange(len(y)), train_size=max_samples,
                                  stratify=y, random_state=seed)
    else:
        take=np.random.default_rng(seed).choice(len(y),max_samples,replace=False)
    return np.asarray(X)[take], y[take], take


def permutation_feature_importances(ensemble, X_test, y_test, feature_names,
                                    repeats=PERMUTATION_REPEATS,
                                    max_samples=MAX_PERMUTATION_SAMPLES,
                                    seed=SEED):
    
    X_sub,y_sub,positions=subsample_for_explanation(X_test,y_test,max_samples,seed)
    if not np.all(np.isfinite(y_sub)):
        raise ValueError('Invalid test labels')
    base = predict_all_models(ensemble, X_sub)
    thresholds = {n: THRESHOLD for n in NAMES}
    thresholds[ENSEMBLE_NAME] = ensemble['decision_threshold']
    baseline = {model:f1_score(y_sub,p>=thresholds[model],zero_division=0)
                for model,p in base.items()}
    rows=[]; rng=np.random.default_rng(seed)
    for j,feat in enumerate(feature_names):
        drops={model:[] for model in base}
        for r in range(repeats):
            Xp=X_sub.copy()
            Xp[:,j]=Xp[rng.permutation(len(Xp)),j]
            ps=predict_all_models(ensemble,Xp)
            for model, probs in ps.items():
                drops[model].append(baseline[model]-
                       f1_score(y_sub,probs>=thresholds[model],zero_division=0))
        for model, values in drops.items():
            rows.append({'Model':model, 'Feature':feat,'Feature_index':j+1,
                         'Baseline_F1_MS':baseline[model],
                         'Mean_delta_F1':float(np.mean(values)),
                         'SD_delta_F1':float(np.std(values,ddof=1)) if repeats>1 else 0.0,
                         'N_test_samples_used':len(y_sub),'Permutation_repeats':repeats})
    return pd.DataFrame(rows), positions


def plot_permutation_importances(df, figure_dir, top_k=20):
    fig, axes=plt.subplots(2,2,figsize=(12,11),constrained_layout=True)
    for ax,model in zip(axes.flat, [ENSEMBLE_NAME]+NAMES):
        d=df[df['Model']==model].nlargest(top_k,'Mean_delta_F1').iloc[::-1]
        ax.barh(d['Feature'],d['Mean_delta_F1'],xerr=d['SD_delta_F1'],capsize=2)
        ax.axvline(0,ls='--',color='gray',lw=1)
        ax.set_title(model)
        ax.set_xlabel('Mean decrease in F1 (after feature permutation)')
        ax.tick_params(axis='y',labelsize=8)
        ax.grid(axis='x',alpha=0.2)
    save_plot(fig,figure_dir/'permutation_importance_F1')


def analyze_feature_contributions(ensemble, X_train, X_test, y_test,
                                  feature_names, output_dir):
    out=Path(output_dir)/'feature_importance'
    figs=Path(output_dir)/'figures'
    out.mkdir(parents=True,exist_ok=True)
    figs.mkdir(parents=True,exist_ok=True)
    built_in=base_feature_importances(ensemble,feature_names)
    built_in.to_csv(out/'base_learners_builtin_importance.csv',index=False)
    plot_base_importances(built_in,figs,TOP_K_FEATURES)

    perm, positions=permutation_feature_importances(ensemble,X_test,y_test,
                                      feature_names)
    perm.to_csv(out/'all_models_permutation_F1.csv',index=False)
    pd.DataFrame({'Zero_based_test_row':positions}).to_csv(
        out/'permutation_test_row_positions.csv',index=False)
    plot_permutation_importances(perm,figs,TOP_K_FEATURES)

    # These weights characterize the fitted meta-learner; due to collinearity
    # they must not be interpreted as causal contributions.
    f_names=[f'Base_probability_{n}' for n in NAMES]
    if ensemble['passthrough']:
        f_names+=['Scaled_'+n for n in feature_names]
    if len(f_names)!=len(ensemble['meta'].coef_.reshape(-1)):
        raise ValueError('Meta feature names mismatch')
    pd.DataFrame({'Meta_input':f_names,
                  'LR_coefficient':ensemble['meta'].coef_.reshape(-1)}).to_csv(
                  out/'stacking_meta_logistic_coefficients.csv',index=False)

    if RUN_SHAP:
        run_stacking_shap(ensemble,X_train,X_test,feature_names,out,figs)
    return built_in,perm




In [12]:
def run_stacking_shap(ensemble, X_train, X_test, feature_names, output_dir, figure_dir,
                      background_size=SHAP_BACKGROUND_SIZE,
                      explanation_size=SHAP_EXPLAIN_SIZE):
    try:
        import shap
    except ImportError as exc:
        raise ImportError('Install SHAP: pip install shap') from exc
    rng=np.random.default_rng(SEED)
    reference=np.asarray(X_train, dtype=float)
    X_to_explain=np.asarray(X_test,dtype=float)
    bg_idx=rng.choice(len(reference),min(background_size,len(reference)),replace=False)
    ex_idx=rng.choice(len(X_to_explain),min(explanation_size,len(X_to_explain)),replace=False)
    # SHAP requires finite values. Impute with *training-only* medians from
    # the fitted preprocessor before building a background/reference distribution.
    imp=ensemble['preprocessor'].named_steps['imputer']
    bg=imp.transform(reference[bg_idx])
    exp=imp.transform(X_to_explain[ex_idx])
    def predictor(z):
        return stacking_probability(ensemble,np.asarray(z,dtype=float))
    masker=shap.maskers.Independent(bg)
    explainer=shap.Explainer(predictor,masker,algorithm='permutation',
                            feature_names=feature_names,seed=SEED)
    shapvalues=explainer(exp,max_evals=2*len(feature_names)+1,
                         batch_size=32,silent=True)
    vals=np.asarray(shapvalues.values)
    if vals.ndim!=2 or vals.shape[1]!=len(feature_names):
        raise ValueError('Unexpected SHAP shape; expected (samples, features)')
    pd.DataFrame(vals,columns=feature_names).to_csv(output_dir/'stacking_SHAP_values.csv',index=False)
    pd.DataFrame({'Feature':feature_names,
                  'Mean_abs_SHAP_probability':np.mean(np.abs(vals),axis=0),
                  'Mean_SHAP_probability':np.mean(vals,axis=0)}).sort_values(
                  'Mean_abs_SHAP_probability',ascending=False).to_csv(
                  output_dir/'stacking_SHAP_mean_absolute.csv',index=False)
    pd.DataFrame({'Zero_based_test_row':ex_idx,
                  'Predicted_MS_probability':predictor(exp)}).to_csv(
                  output_dir/'stacking_SHAP_sample_rows.csv',index=False)
    shap.plots.beeswarm(shapvalues,max_display=TOP_K_FEATURES,show=False)
    plt.gcf().savefig(figure_dir/'stacking_SHAP_beeswarm.png',dpi=IMAGE_DPI,bbox_inches='tight')
    plt.close('all')
    shap.plots.bar(shapvalues,max_display=TOP_K_FEATURES,show=False)
    plt.gcf().savefig(figure_dir/'stacking_SHAP_bar.png',dpi=IMAGE_DPI,bbox_inches='tight')
    plt.close('all')
    shap.plots.waterfall(shapvalues[0],max_display=TOP_K_FEATURES,show=False)
    plt.gcf().savefig(figure_dir/'stacking_SHAP_waterfall_example.png',
                      dpi=IMAGE_DPI,bbox_inches='tight')
    plt.close('all')
    return shapvalues




In [13]:
# ============================================================
# Original requested feature_importances_ interfaces (with adapter)
# ============================================================
class FeatureImportanceView:
    """Make native boosters look like sklearn models for feature-importance use only."""
    def __init__(self, booster, kind, feature_names):
        self.booster = booster
        self.kind = kind
        self.feature_names_ = list(feature_names)
        n = len(self.feature_names_)
        if kind == 'xgb':
            score = booster.get_score(importance_type='gain')
            values = [score.get(f'f{i}', 0.0) for i in range(n)]
        elif kind == 'lgb':
            values = booster.feature_importance(importance_type='gain')
        elif kind == 'cat':
            values = booster.get_feature_importance(type='FeatureImportance')
        else:
            raise ValueError(kind)
        self.feature_importances_ = np.asarray(values, dtype=float)

    def get_feature_importance(self):
        if self.kind != 'cat':
            raise ValueError('Only CatBoost exposes get_feature_importance here')
        return self.feature_importances_.copy()


def feature_importance_xgboost(model):
    result = pd.DataFrame(model.feature_importances_,
                          index=model.feature_names_,
                          columns=['FeatureImportance'])
    return result.sort_values('FeatureImportance', ascending=False)


def feature_importance_lightgbm(model):
    result = pd.DataFrame(model.feature_importances_,
                          index=model.feature_names_,
                          columns=['FeatureImportance'])
    return result.sort_values('FeatureImportance', ascending=False)


def feature_importance_catboost(model):
    result = pd.DataFrame(model.get_feature_importance(),
                          index=model.feature_names_,
                          columns=['FeatureImportance'])
    return result.sort_values('FeatureImportance', ascending=False)




# ============================================================
# Per-iteration train/validation focal loss and accuracy
# ============================================================






In [14]:

from sklearn.metrics import brier_score_loss


def bo_sample_params(trial, name):
    """Sample an actual configuration of one focal-loss booster in the stacked path."""
    if name == 'xgb':
        return dict(max_depth=trial.suggest_int('max_depth', 1, 4),
                    learning_rate=trial.suggest_float('learning_rate', 0.025, 0.26, log=True),
                    subsample=trial.suggest_float('subsample', 0.70, 1.0),
                    reg_lambda=trial.suggest_float('reg_lambda', 0.05, 5.0, log=True),
                    min_child_weight=trial.suggest_float('min_child_weight', 0.002, 0.20, log=True),
                    num_boost_round=trial.suggest_int('num_boost_round', 20, 100, step=10))
    if name == 'lgb':
        return dict(max_depth=trial.suggest_int('max_depth', 2, 4),
                    num_leaves=trial.suggest_int('num_leaves', 3, 12),
                    learning_rate=trial.suggest_float('learning_rate', 0.025, 0.26, log=True),
                    feature_fraction=trial.suggest_float('feature_fraction', 0.70, 1.0),
                    num_boost_round=trial.suggest_int('num_boost_round', 20, 100, step=10))
    if name == 'cat':
        return dict(depth=trial.suggest_int('depth', 2, 4),
                    learning_rate=trial.suggest_float('learning_rate', 0.025, 0.26, log=True),
                    l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 0.5, 8.0, log=True),
                    num_boost_round=trial.suggest_int('num_boost_round', 20, 100, step=10))
    raise ValueError(name)


def get_training_cv(X_train, y_train, groups_train=None):
    """Build leakage-safe training folds including fold-specific preprocessing."""
    if groups_train is None:
        iterator = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X_train, y_train)
    else:
        iterator = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True,
                                        random_state=SEED).split(X_train, y_train, groups_train)
    folds = []
    for fi, vi in iterator:
        if len(np.unique(y_train[fi])) < 2 or len(np.unique(y_train[vi])) < 2:
            raise ValueError('Training CV fold has a single class. Provide more independent events.')
        pp_fold = new_preprocessor()
        folds.append((fi, vi, pp_fold.fit_transform(X_train[fi]),
                      pp_fold.transform(X_train[vi])))
    return folds


def optimize_base_for_stack(name, y_train, folds):
    try:
        import optuna
    except ImportError as e:
        raise ImportError('Install optuna: pip install optuna. '
                          'Only the stacking branch uses Bayesian optimization.') from e
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    trial_rows=[]
    def objective(trial):
        p = bo_sample_params(trial, name)
        fold_values=[]
        for fi, vi, Xf, Xo in folds:
            model, _ = fit_base(name, p, Xf, y_train[fi])
            prob = np.clip(predict_base(name, model, Xo), 1e-7, 1-1e-7)
            yt = y_train[vi]
            ap = average_precision_score(yt, prob)
            auc = roc_auc_score(yt, prob)
            f1 = f1_score(yt, prob >= THRESHOLD, zero_division=0)
            brier = brier_score_loss(yt, prob)
            fold_values.append(0.35*ap + 0.25*auc + 0.25*f1 + 0.15*(1-brier))
        val = float(np.mean(fold_values))
        trial_rows.append({'Base':name,'Trial':trial.number,'Train_CV_score':val,
                           **p})
        return val
    sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=max(2,N_INIT))
    study=optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(objective, n_trials=N_INIT+N_BO_ITER, show_progress_bar=False)
    pd.DataFrame(trial_rows).to_csv(OUTPUT_DIR / f'BO_train_CV_{name}_trials.csv', index=False)
    print(f'BO {name.upper()} training-CV best={study.best_value:.4f}, trials={len(study.trials)}')
    return {**study.best_params}, float(study.best_value)


def get_meta_validation_candidate(oof, y_train, valid_scores, y_valid):
    """Choose LR settings on VALIDATION only; no predictions from test are referenced."""
    candidates=[]
    for class_weight in (None,'balanced'):
        for C in (0.03, 0.10, 0.30, 1.0, 3.0, 10.0):
            meta=LogisticRegression(C=C, class_weight=class_weight,
                                    max_iter=2500, random_state=SEED)
            meta.fit(oof, y_train)
            probs=meta.predict_proba(valid_scores)[:,1]
            ap=average_precision_score(y_valid, probs)
            auc=roc_auc_score(y_valid, probs)
            brier=brier_score_loss(y_valid, probs)
            for th in np.arange(0.20,0.801,0.025):
                f1=f1_score(y_valid,probs >= th,zero_division=0)
                balanced=balanced_accuracy_score(y_valid,probs >= th)
                # F1 dominates; probability-quality breaks ties and avoids all-positive solutions
                selection=0.60*f1+0.15*balanced+0.15*ap+0.05*auc+0.05*(1-brier)
                candidates.append({'C':C,'class_weight':class_weight,
                                   'threshold':float(th), 'Validation_F1':f1,
                                   'Validation_AUC':auc, 'Validation_AP':ap,
                                   'Validation_Brier':brier, 'Score':selection})
    df=pd.DataFrame(candidates).sort_values(['Score','Validation_Brier'],ascending=[False,True])
    df.to_csv(OUTPUT_DIR / 'stacking_meta_validation_candidates.csv', index=False)
    winner=df.iloc[0]
    # pandas may convert None to NaN, so check explicitly
    cw=winner['class_weight']
    if pd.isna(cw): cw=None
    meta=LogisticRegression(C=float(winner['C']),class_weight=cw,
                            max_iter=2500,random_state=SEED)
    meta.fit(oof,y_train)
    return meta,float(winner['threshold']),dict(winner)


def fit_stacking(X_train,y_train,X_valid,y_valid,groups_train=None):
    """Independent baselines + Bayesian-optimized stacking branch."""
    folds=get_training_cv(X_train,y_train,groups_train)
    pp=new_preprocessor()
    Xt=pp.fit_transform(X_train)
    Xv=pp.transform(X_valid)

    # Independent standalone models are NEVER Bayesian-optimized.
    baseline={}
    for name in NAMES:
        baseline[name],_=fit_base(name,PARAMS[name],Xt,y_train)
    print('Standalone parameter models fitted (no Bayesian optimization).')

    # Optimize only the model copies used inside the BO-IGBT stacking branch.
    tuned={}
    bo_train_scores={}
    for name in NAMES:
        tuned[name],bo_train_scores[name]=optimize_base_for_stack(name,y_train,folds)

    # 3-fold OOF input to meta-classifier; fold-specific preprocessing.
    oof=np.full((len(y_train),3),np.nan)
    for fold_num,(fi,vi,Xf,Xo) in enumerate(folds,1):
        for j,name in enumerate(NAMES):
            est,_=fit_base(name,tuned[name],Xf,y_train[fi])
            oof[vi,j]=predict_base(name,est,Xo)
        print(f'BO-IGBT OOF fold {fold_num}/{len(folds)} completed.')
    if not np.isfinite(oof).all():
        raise ValueError('OOF predictions missing.')

    tuned_models={}
    for name in NAMES:
        tuned_models[name],_=fit_base(name,tuned[name],Xt,y_train)
    valid_scores=np.column_stack([predict_base(n,tuned_models[n],Xv) for n in NAMES])
    meta, threshold,selected=get_meta_validation_candidate(oof,y_train,valid_scores,y_valid)
    print(f'BO-IGBT meta: C={selected["C"]}, class_weight={selected["class_weight"]}, '
          f'validation-selected threshold={threshold:.3f}')
    (OUTPUT_DIR/'meta_best_validation_setting.json').write_text(json.dumps({
        'C':float(selected['C']),
        'class_weight':None if pd.isna(selected['class_weight']) else str(selected['class_weight']),
        'threshold':threshold, 'Validation_F1':float(selected['Validation_F1'])},
        indent=2), encoding='utf-8')
    return {'preprocessor':pp,'baseline':baseline,'base':tuned_models,
            'tuned':tuned,'meta':meta,'passthrough':False,
            'train_oof_base':oof,'decision_threshold':threshold,
            'bayes_opt_enabled':True,'bo_train_cv_scores':bo_train_scores}


def predict_all_models(ensemble,X):
    Xt=ensemble['preprocessor'].transform(X)
    # Single-model columns are from standalone models, NOT tuned ensemble models.
    out={n:predict_base(n,ensemble['baseline'][n],Xt) for n in NAMES}
    stack_inputs=np.column_stack([predict_base(n,ensemble['base'][n],Xt) for n in NAMES])
    out[ENSEMBLE_NAME]=ensemble['meta'].predict_proba(stack_inputs)[:,1]
    return out


def export_requested_feature_importance(ens, feature_names):
    out=OUTPUT_DIR/'feature_importance'
    out.mkdir(parents=True,exist_ok=True)
    models={
        'XGBoost':FeatureImportanceView(ens['baseline']['xgb'],'xgb',feature_names),
        'LightGBM':FeatureImportanceView(ens['baseline']['lgb'],'lgb',feature_names),
        'CatBoost':FeatureImportanceView(ens['baseline']['cat'],'cat',feature_names)
    }
    results={'XGBoost':feature_importance_xgboost(models['XGBoost']),
             'LightGBM':feature_importance_lightgbm(models['LightGBM']),
             'CatBoost':feature_importance_catboost(models['CatBoost'])}
    fig,axarr=plt.subplots(1,3,figsize=(15.5,5),constrained_layout=True)
    for ax,(name,df) in zip(axarr,results.items()):
        df.index.name='Feature'
        df.to_csv(out/f'{name}_feature_importance.csv')
        display=df.head(min(12,len(df))).iloc[::-1]
        ax.barh(display.index,display.FeatureImportance)
        ax.set_title(name)
        ax.set_xlabel('Model-specific feature importance')
        ax.grid(axis='x',alpha=0.22)
    save_plot(fig,OUTPUT_DIR/'figures'/'feature_importance')
    
    for n in NAMES:
        v=FeatureImportanceView(ens['base'][n],n,feature_names)
        pd.DataFrame({'Feature':feature_names,'Importance':v.feature_importances_}).sort_values(
            'Importance',ascending=False).to_csv(out/f'{n}_BO_component_importance.csv',index=False)
    pd.DataFrame({'Meta_input':[f'P_MS_{n}' for n in NAMES],
                  'LR_coefficient':ens['meta'].coef_.ravel()}).to_csv(
        out/'BO_IGBT_meta_coefficients.csv',index=False)
    return results


def plot_training_validation_curves(ens, X_train,y_train,X_valid,y_valid):
    
    pp=ens['preprocessor']
    Xt,Xv=pp.transform(X_train),pp.transform(X_valid)
    history=[]
    curves={}
    for name in NAMES:
        model=ens['baseline'][name]
        if name=='xgb': rounds=int(model.num_boosted_rounds())
        elif name=='lgb': rounds=int(model.current_iteration())
        else: rounds=int(model.tree_count_)
        rows=[]
        for k in range(1,rounds+1):
            p_tr=predict_base(name,model,Xt,n_trees=k)
            p_va=predict_base(name,model,Xv,n_trees=k)
            loss_tr=float(np.mean(FOCAL.loss(logit(np.clip(p_tr,1e-12,1-1e-12)),y_train)))
            loss_va=float(np.mean(FOCAL.loss(logit(np.clip(p_va,1e-12,1-1e-12)),y_valid)))
            rows.append({'Model':name,'Iteration':k,'Train_Focal_Loss':loss_tr,
                'Validation_Focal_Loss':loss_va,
                'Train_Accuracy':accuracy_score(y_train,p_tr >= THRESHOLD),
                'Validation_Accuracy':accuracy_score(y_valid,p_va >= THRESHOLD)})
        curves[name]=pd.DataFrame(rows)
        history.extend(rows)
    df=pd.DataFrame(history)
    df.to_csv(OUTPUT_DIR/'train_validation_learning_curves.csv',index=False)
    fig,axes=plt.subplots(3,2,figsize=(12.5,10.5),constrained_layout=True)
    for i,name in enumerate(NAMES):
        d=curves[name]
        ax1,ax2=axes[i]
        ax1.plot(d.Iteration,d.Train_Focal_Loss,label='Train')
        ax1.plot(d.Iteration,d.Validation_Focal_Loss,label='Validation')
        ax1.set(title=f'{name.upper()} : focal loss',xlabel='Iteration',ylabel='Focal loss')
        ax2.plot(d.Iteration,d.Train_Accuracy,label='Train')
        ax2.plot(d.Iteration,d.Validation_Accuracy,label='Validation')
        ax2.set(title=f'{name.upper()} : accuracy',xlabel='Iteration',ylabel='Accuracy',ylim=(-0.03,1.03))
        for ax in (ax1,ax2):ax.legend(fontsize=8);ax.grid(alpha=.2)
    save_plot(fig,OUTPUT_DIR/'figures'/'train_validation_loss_accuracy')
    return df



def run_main():
    X,y,groups=load_dataset(DATA_DIR)
    feature_names=load_feature_names(DATA_DIR,X.shape[1])
    train_id,valid_id,test_id=split_data(X,y,groups)
    ensemble=fit_stacking(X[train_id],y[train_id],X[valid_id],y[valid_id],
        groups_train=groups[train_id] if groups is not None else None)
    split_map={'train':train_id,'validation':valid_id,'test':test_id}
    metrics=evaluate_all_splits(ensemble,X,y,split_map,groups=groups)
    metrics.to_csv(OUTPUT_DIR/'all_models_all_splits_metrics.csv',index=False)
    metrics[metrics['Model']==ENSEMBLE_NAME].to_csv(OUTPUT_DIR/'main_scores.csv',index=False)
    save_train_oof_base_results(ensemble,y[train_id],train_id,groups)
    save_split_summary(y,split_map,groups)
    np.savez_compressed(OUTPUT_DIR/'split_indices.npz',train=train_id,
                        validation=valid_id,test=test_id)
    (OUTPUT_DIR/'parameters_and_bo.json').write_text(json.dumps({
        'standalone':PARAMS,'bo_stacking_components':ensemble['tuned'],
        'meta_threshold':ensemble['decision_threshold'],
        'bo_train_cv_scores':ensemble['bo_train_cv_scores'],
        'data_source':'real' if (DATA_DIR/'Features_matrix.mat').exists() else 'synthetic_demo'
    },indent=2),encoding='utf-8')
    plot_training_validation_curves(ensemble,X[train_id],y[train_id],X[valid_id],y[valid_id])
    export_requested_feature_importance(ensemble,feature_names)
    if RUN_EXPLANATIONS:
        analyze_feature_contributions(ensemble,X[train_id],X[test_id],y[test_id],
            feature_names,OUTPUT_DIR)
    print(f'Complete; results in: {OUTPUT_DIR.resolve()}')
    return ensemble,(X,y,groups,train_id,valid_id,test_id),metrics


if __name__=='__main__':
    check_focal_derivatives()
    ensemble,split_info,results=run_main()
    print(results[['Split','Model','N','MS','NS','Accuracy','F1_MS']].to_string(index=False))


Focal derivative finite-difference tests passed.
Loaded REAL data: F:\Research_Work\Academic_Papers\Publish_Paper\Catboot_Classification\BO_IGBT\Data_sets\MS_NS_TrainData\Features_matrix.mat and F:\Research_Work\Academic_Papers\Publish_Paper\Catboot_Classification\BO_IGBT\Data_sets\MS_NS_TrainData\Label.mat
Samples=21 | MS=7 | NS=14 | features=14
     train: N=13 | MS= 5 | NS= 8
validation: N= 4 | MS= 1 | NS= 3
      test: N= 4 | MS= 1 | NS= 3


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7092\2265204007.py:67: UserWarning: Sample-wise split: correlated segments of the same event could leak across sets. Provide Group.mat where available.
  warnings.warn('Sample-wise split: correlated segments of the same event '
D:\ProgramData\anaconda3\envs\ML\lib\site-packages\catboost\core.py:2321: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


Standalone parameter models fitted (no Bayesian optimization).


D:\ProgramData\anaconda3\envs\ML\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BO XGB training-CV best=0.6709, trials=12
BO LGB training-CV best=0.8552, trials=12


D:\ProgramData\anaconda3\envs\ML\lib\site-packages\catboost\core.py:2321: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)
D:\ProgramData\anaconda3\envs\ML\lib\site-packages\catboost\core.py:2321: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)
D:\ProgramData\anaconda3\envs\ML\lib\site-packages\catboost\core.py:2321: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)
D:\ProgramData\anaconda3\envs\ML\lib\site-packages\catboost\core.py:2321: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)
D:\ProgramData\anaconda3\envs\ML\lib\site-packages\catboost\core.py:2321: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)
D:\ProgramData\anaconda3\envs\ML\lib\site-packages\catboost\core.py:2321: U

... (5 outputs left out)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


test              | xgb      | F1_MS=1.0000 | ROC_AUC=1.0000 | N=4
test              | lgb      | F1_MS=1.0000 | ROC_AUC=1.0000 | N=4
test              | cat      | F1_MS=1.0000 | ROC_AUC=1.0000 | N=4
test              | BO-IGBT  | F1_MS=1.0000 | ROC_AUC=1.0000 | N=4


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Complete; results in: F:\Research_Work\Academic_Papers\Publish_Paper\Catboot_Classification\BO_IGBT\Results_BO_IGBT
     Split   Model  N  MS  NS  Accuracy    F1_MS
     train     xgb 13   5   8      1.00 1.000000
     train     lgb 13   5   8      1.00 1.000000
     train     cat 13   5   8      1.00 1.000000
     train BO-IGBT 13   5   8      1.00 1.000000
validation     xgb  4   1   3      0.75 0.666667
validation     lgb  4   1   3      0.75 0.666667
validation     cat  4   1   3      1.00 1.000000
validation BO-IGBT  4   1   3      1.00 1.000000
      test     xgb  4   1   3      1.00 1.000000
      test     lgb  4   1   3      1.00 1.000000
      test     cat  4   1   3      1.00 1.000000
      test BO-IGBT  4   1   3      1.00 1.000000
